# Acceptance test - Dask

This notebook allows demonstrating the use of the Dask cluster.

It includes the same code as the integration test proposed by the Python project template.

In [ ]:
import os
import dask.array as da
from dask_gateway import Gateway

# The following Dask environment variables are required
# to connect to the Dask cluster.
# This cell is validating that the variables are made available by default.
daskEnvVars = [
    "DASK_GATEWAY__ADDRESS",
    "DASK_GATEWAY__AUTH__TYPE",
    "DASK_GATEWAY__PROXY_ADDRESS",
    "DASK_GATEWAY__PUBLIC_ADDRESS",
    "JUPYTERHUB_API_TOKEN",
]

for var in daskEnvVars:
    try:
        value = os.environ[var]
        print("INFO:  The '" + var + "' is set to '" + value + "'")
    except KeyError:
        print("ERROR: The '" + var + "' environment variable is missing")

## Set up the cluster

In [ ]:
gateway = Gateway()

# The Dask cluster is deployed using a default image.
# However, it is possible to specify the image to be used for the
# scheduler and workers when creating the cluster.
# In this example, the 'project template' image is used.
# Please use the default one or specify your own dedicated image.
image = "registry.eopf.copernicus.eu/sde/project-template:latest"

print("Creating Dask cluster, using image:", image)
cluster = gateway.new_cluster(image=image)

# Use two workers
cluster.scale(2)
client = cluster.get_client()

# Display cluster information
#
# The following lines provide some information on the cluster that has been created.
print("Client:", client)
print("Cluster:", cluster)
print("Scheduler info:", client.scheduler_info())
print("Dashboard:", "https://jhub-sde.eopf.copernicus.eu" + cluster.dashboard_link)

In [ ]:
# Run some dummy code in the cluster
x = da.random.random((10000, 10000), chunks=(10000, 1000))
y = x + x.T
z = y[::2, 5000:].mean(axis=1)

print("Starting computing ...")
array = z.compute()
print("Computing terminated")

## Close the cluster

In [ ]:
client.close()

In [ ]:
cluster.close()

In [ ]:
gateway.close()